# Custom middleware

Xây dựng custom middleware bằng cách triển khai các hook chạy tại những thời điểm cụ thể trong luồng thực thi của agent.

## Hook

Middleware cung cấp hai kiểu hook để can thiệp vào quá trình thực thi của agent:

* **Node-style hook**: Chạy tuần tự tại các điểm thực thi cụ thể.
* **Wrap-style hook**: Chạy bao quanh mỗi lời gọi model hoặc tool.

### Node-style hook

Chạy tuần tự tại các điểm thực thi cụ thể. Sử dụng cho quá trình logging, xác thực và cập nhật state.

Chọn các hook mà middleware của bạn cần. Bạn có thể chọn giữa node-style hooks và wrap-style hooks.

**Node-style hook** chạy tại các điểm thực thi cụ thể:

| Hook           | Khi nào chạy                                     |
| -------------- | ------------------------------------------------ |
| `before_agent` | Trước khi agent bắt đầu (một lần mỗi lần gọi)    |
| `before_model` | Trước mỗi lời gọi model                          |
| `after_model`  | Sau mỗi phản hồi của model                       |
| `after_agent`  | Sau khi agent hoàn thành (một lần mỗi lần gọi)   |

**Wrap-style hook** chạy bao quanh mỗi lời gọi, cho phép bạn kiểm soát quá trình thực thi:

| Hook              | Khi nào chạy                  |
| ----------------- | ----------------------------- |
| `wrap_model_call` | Bao quanh mỗi lời gọi model   |
| `wrap_tool_call`  | Bao quanh mỗi lời gọi tool    |

In [ ]:
from langchain.agents.middleware import before_model, after_model, AgentState
from langchain.messages import AIMessage
from langgraph.runtime import Runtime
from typing import Any


@before_model(can_jump_to=["end"])
def check_message_limit(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    if len(state["messages"]) >= 50:
        return {
            "messages": [AIMessage("Đã đạt giới hạn hội thoại.")],
            "jump_to": "end"
        }
    return None

@after_model
def log_response(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(f"Model trả về: {state['messages'][-1].content}")
    return None

### Wrap-style hook

Can thiệp vào quá trình thực thi và kiểm soát thời điểm handler được gọi. Sử dụng cho cơ chế retry, caching và biến đổi dữ liệu.

Bạn có thể quyết định xem handler không được gọi lần nào (short-circuit), một lần (luồng bình thường) hay nhiều lần (logic retry).

**Các hook có sẵn:**

* `wrap_model_call` - Bao quanh mỗi lời gọi model
* `wrap_tool_call` - Bao quanh mỗi lời gọi tool

**Ví dụ:**

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable


@wrap_model_call
def retry_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    for attempt in range(3):
        try:
            return handler(request)
        except Exception as e:
            if attempt == 2:
                raise
            print(f"Thử lại lần {attempt + 1}/3 sau lỗi: {e}")

## Cập nhật state

Cả node-style và wrap-style hook đều có thể cập nhật agent state. Cơ chế hoạt động của chúng có sự khác biệt:

* **Node-style hook** (`before_agent`, `before_model`, `after_model`, `after_agent`): Trả về trực tiếp một dict. Dict này được áp dụng vào agent state thông qua các reducer của graph.
* **Wrap-style hook** (`wrap_model_call`, `wrap_tool_call`): Đối với lời gọi model, trả về [`ExtendedModelResponse`](https://reference.langchain.com/python/langchain/agents/middleware/types/ExtendedModelResponse) kèm theo một [`Command`](https://reference.langchain.com/python/langgraph/types/Command) để tiêm (inject) các cập nhật state cùng với phản hồi của model. Đối với lời gọi tool, trả về trực tiếp một [`Command`](https://reference.langchain.com/python/langgraph/types/Command). Sử dụng cơ chế này khi bạn cần theo dõi hoặc cập nhật state dựa trên logic chạy trong lúc gọi model hoặc tool, chẳng hạn như các điểm kích hoạt tóm tắt, metadata mức độ sử dụng hoặc các trường custom tính toán từ request/response.

### Node-style hook

Trả về một dict từ node-style hook để hợp nhất các bản cập nhật vào agent state. Các key trong dict sẽ tương ứng với các trường trong state.

In [ ]:
from langchain.agents.middleware import after_model, AgentState
from langgraph.runtime import Runtime
from typing import Any
from typing_extensions import NotRequired


class TrackingState(AgentState):
    model_call_count: NotRequired[int]


@after_model(state_schema=TrackingState)
def increment_after_model(state: TrackingState, runtime: Runtime) -> dict[str, Any] | None:
    return {"model_call_count": state.get("model_call_count", 0) + 1}

### Wrap-style hooks

Trả về một [`ExtendedModelResponse`](https://reference.langchain.com/python/langchain/agents/middleware/types/ExtendedModelResponse) đi kèm với [`Command`](https://reference.langchain.com/python/langgraph/types/Command) từ `wrap_model_call` để tiêm các bản cập nhật state từ layer gọi model:

```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
from typing import Callable
from langchain.agents.middleware import (
    wrap_model_call,
    ModelRequest,
    ModelResponse,
    AgentState,
    ExtendedModelResponse
)
from langgraph.types import Command
from typing_extensions import NotRequired

class UsageTrackingState(AgentState):
    """Agent state có tính năng theo dõi mức sử dụng token."""

    last_model_call_tokens: NotRequired[int]


@wrap_model_call(state_schema=UsageTrackingState)
def track_usage(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ExtendedModelResponse:
    response = handler(request)
    return ExtendedModelResponse(
        model_response=response,
        command=Command(update={"last_model_call_tokens": 150}),
    )
```

Các [`Command`](https://reference.langchain.com/python/langgraph/types/Command) sẽ truyền qua các reducer của graph, do đó các bản cập nhật được áp dụng một cách chính xác và các message sẽ mang tính cộng dồn (additive) thay vì ghi đè lên state hiện tại.

#### Hợp nhất với nhiều middleware

Khi nhiều layer middleware trả về `ExtendedModelResponse`, các command của chúng sẽ được kết hợp lại:

* **Command được áp dụng thông qua reducer:** Mỗi `Command` trở thành một bản cập nhật state riêng biệt. Đối với các message, điều này có nghĩa là chúng sẽ mang tính chất cộng dồn.
* **Lớp ngoài cùng ưu tiên khi có xung đột (Outer wins):** Đối với các trường state không sử dụng reducer, các command được áp dụng từ trong ra ngoài (inner-first, then outer). Giá trị của middleware nằm ngoài cùng sẽ chiếm ưu thế khi có xung đột tại cùng một key.
* **An toàn khi thử lại (Retry-safe):** Nếu middleware bên ngoài triển khai logic có thể dẫn đến việc gọi lại `handler()` nhiều lần (ví dụ: logic retry), các command từ các lần gọi trước đó sẽ bị loại bỏ.

```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
from typing import Annotated, Callable

from langchain.agents.middleware import (
    AgentMiddleware,
    AgentState,
    ExtendedModelResponse,
    ModelRequest,
    ModelResponse,
)
from langchain.messages import SystemMessage
from langgraph.types import Command
from typing_extensions import NotRequired


def _last_wins(_a: str, b: str) -> str:
    """Reducer: người ghi cuối cùng sẽ thắng (lớp ngoài ghi đè lớp trong)."""
    return b


class CustomMiddlewareState(AgentState):
    """Agent state: trace_layer sử dụng last-wins (lớp ngoài thắng), messages sử dụng additive reducer (cộng dồn)."""

    # Trường không dùng reducer với tính chất last-wins: cả hai middleware đều ghi; giá trị ngoài cùng chiến thắng
    trace_layer: NotRequired[Annotated[str, _last_wins]]


class OuterMiddleware(AgentMiddleware):
    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ExtendedModelResponse:
        response = handler(request)
        return ExtendedModelResponse(
            model_response=response,
            command=Command(update={
                "trace_layer": "outer",
                "messages": [SystemMessage(content="[Lớp ngoài đã chạy]")],
            }),
        )


class InnerMiddleware(AgentMiddleware):
    """Thêm trace_layer và message. Lớp ngoài thêm vào các key tương tự; trace_layer: lớp ngoài thắng, messages: cộng dồn."""

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ):
        response = handler(request)
        return ExtendedModelResponse(
            model_response=response,
            command=Command(update={
                "trace_layer": "inner",
                "messages": [SystemMessage(content="[Lớp trong đã chạy]")],
            }),
        )
```

## Tạo middleware

Bạn có thể tạo middleware theo hai cách:

<CardGroup cols={2}>
  <Card title="Decorator-based middleware" icon="at" href="#decorator-based-middleware">
    Nhanh chóng và đơn giản cho middleware dùng một hook. Sử dụng decorator để bao bọc các hàm riêng lẻ.
  </Card>

  <Card title="Class-based middleware" icon="braces" href="#class-based-middleware">
    Mạnh mẽ hơn cho các middleware phức tạp với nhiều hook hoặc cấu hình.
  </Card>
</CardGroup>

### Decorator-based middleware

Nhanh chóng và đơn giản cho middleware dùng một hook. Sử dụng decorator để bao bọc (wrap) các hàm riêng lẻ.

**Các decorator khả dụng:**

**Node-style:**

* [`@before_agent`](https://reference.langchain.com/python/langchain/agents/middleware/types/before_agent) - Chạy trước khi agent bắt đầu (một lần mỗi lần gọi)
* [`@before_model`](https://reference.langchain.com/python/langchain/agents/middleware/types/before_model) - Chạy trước mỗi lời gọi model
* [`@after_model`](https://reference.langchain.com/python/langchain/agents/middleware/types/after_model) - Chạy sau mỗi phản hồi của model
* [`@after_agent`](https://reference.langchain.com/python/langchain/agents/middleware/types/after_agent) - Chạy sau khi agent hoàn thành (một lần mỗi lần gọi)

**Wrap-style:**

* [`@wrap_model_call`](https://reference.langchain.com/python/langchain/agents/middleware/types/wrap_model_call) - Bao bọc mỗi lời gọi model với logic tùy chỉnh
* [`@wrap_tool_call`](https://reference.langchain.com/python/langchain/agents/middleware/types/wrap_tool_call) - Bao bọc mỗi lời gọi tool với logic tùy chỉnh

**Hỗ trợ tiện ích (Convenience):**

* [`@dynamic_prompt`](https://reference.langchain.com/python/langchain/agents/middleware/types/dynamic_prompt) - Tạo ra các system prompt động

**Ví dụ:**

```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
from langchain.agents.middleware import (
    before_model,
    wrap_model_call,
    AgentState,
    ModelRequest,
    ModelResponse,
)
from langchain.agents import create_agent
from langgraph.runtime import Runtime
from typing import Any, Callable


@before_model
def log_before_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(f"Chuẩn bị gọi model với {len(state['messages'])} message")
    return None

@wrap_model_call
def retry_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    for attempt in range(3):
        try:
            return handler(request)
        except Exception as e:
            if attempt == 2:
                raise
            print(f"Thử lại lần {attempt + 1}/3 sau lỗi: {e}")

agent = create_agent(
    model="gpt-5.5",
    middleware=[log_before_model, retry_model],
    tools=[...],
)
```

**Khi nào nên dùng decorator:**

* Chỉ cần một hook duy nhất
* Không có cấu hình phức tạp
* Prototype nhanh chóng

### Class-based middleware

Mạnh mẽ hơn cho các middleware phức tạp với nhiều hook hoặc cấu hình. Sử dụng class khi bạn cần khai báo cả phiên bản đồng bộ (sync) và bất đồng bộ (async) cho cùng một hook, hoặc khi bạn muốn kết hợp nhiều hook vào một middleware duy nhất.

:::python
Một subclass của `AgentMiddleware` có thể khai báo ba thuộc tính class mà agent factory sẽ tự động nhận diện tại thời điểm biên dịch (compile time):

* `state_schema` — mở rộng agent state với các trường custom. Xem thêm [Custom state schema](#custom-state-schema).
* `tools` — đăng ký các tool bổ sung đi kèm với middleware (ví dụ: `write_todos` trên middleware danh sách to-do).
* `transformers` — đăng ký các stream transformer factory có nhận thức về phạm vi (scope-aware). Xem thêm [Custom stream transformers](#custom-stream-transformers).
:::

**Ví dụ:**

```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
from langchain.agents.middleware import (
    AgentMiddleware,
    AgentState,
    ModelRequest,
    ModelResponse,
)
from langgraph.runtime import Runtime
from typing import Any, Callable

class LoggingMiddleware(AgentMiddleware):
    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        print(f"Chuẩn bị gọi model với {len(state['messages'])} message")
        return None

    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        print(f"Model trả về: {state['messages'][-1].content}")
        return None

    async def abefore_model(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        # Phiên bản bất đồng bộ của before_model
        return None

    async def aafter_model(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        # Phiên bản bất đồng bộ của after_model
        print(f"Model trả về: {state['messages'][-1].content}")
        return None


agent = create_agent(
    model="gpt-5.5",
    middleware=[LoggingMiddleware()],
    tools=[...],
)
```

**Khi nào nên dùng class:**

* Cần khai báo cả implementation sync và async cho cùng một hook
* Cần sử dụng nhiều hook trong một middleware duy nhất
* Yêu cầu cấu hình phức tạp (ví dụ: ngưỡng có thể tùy chỉnh, custom model)
* Tái sử dụng trên nhiều project với cấu hình được truyền vào thời điểm khởi tạo (init-time)

## Custom state schema

Nếu middleware của bạn cần theo dõi state xuyên suốt các hook, middleware có thể mở rộng state của agent với các thuộc tính custom. Điều này cho phép middleware:

* **Theo dõi state xuyên suốt quá trình thực thi**: Duy trì các bộ đếm, cờ (flags) hoặc các giá trị khác tồn tại trong suốt vòng đời thực thi của agent

* **Chia sẻ dữ liệu giữa các hook**: Truyền thông tin từ `before_model` sang `after_model` hoặc giữa các instance middleware khác nhau

* **Triển khai các concern cắt ngang (cross-cutting concerns)**: Thêm chức năng như rate limiting, theo dõi mức sử dụng, user context, hoặc audit logging mà không cần chỉnh sửa logic cốt lõi của agent

* **Đưa ra các quyết định có điều kiện**: Sử dụng state đã tích lũy để xác định xem có nên tiếp tục thực thi, nhảy đến các node khác hay thay đổi hành vi động hay không

<Tabs>
  <Tab title="Decorator">
    ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
    from langchain.agents import create_agent
    from langchain.messages import HumanMessage
    from langchain.agents.middleware import AgentState, before_model, after_model
    from typing_extensions import NotRequired
    from typing import Any
    from langgraph.runtime import Runtime


    class CustomState(AgentState):
        model_call_count: NotRequired[int]
        user_id: NotRequired[str]


    @before_model(state_schema=CustomState, can_jump_to=["end"])
    def check_call_limit(state: CustomState, runtime: Runtime) -> dict[str, Any] | None:
        count = state.get("model_call_count", 0)
        if count > 10:
            return {"jump_to": "end"}
        return None


    @after_model(state_schema=CustomState)
    def increment_counter(state: CustomState, runtime: Runtime) -> dict[str, Any] | None:
        return {"model_call_count": state.get("model_call_count", 0) + 1}


    agent = create_agent(
        model="gpt-5.5",
        middleware=[check_call_limit, increment_counter],
        tools=[],
    )

    # Gọi (invoke) cùng với custom state
    result = agent.invoke({
        "messages": [HumanMessage("Xin chào")],
        "model_call_count": 0,
        "user_id": "user-123",
    })
    ```
  </Tab>
</Tabs>

## Custom stream transformers

<Note>Các transformer được đăng ký trên middleware yêu cầu `langchain>=1.3.2`.</Note>

Middleware có thể đăng ký các stream transformer factory nhằm project (ánh xạ) các sự kiện từ luồng trực tiếp của agent sang các extension channel có kiểu dữ liệu (typed extension channels). Điều này rất hữu ích để hiển thị các bộ đếm, side-channel artifacts, output một phần (partial outputs), hoặc loại bỏ thông tin nhạy cảm ở cấp độ mạng (wire-level redaction) mà không bị phụ thuộc chặt chẽ vào các phép chiếu có sẵn của framework.

Tại thời điểm biên dịch, các factory do middleware đăng ký sẽ được hợp nhất với bất cứ thứ gì người gọi truyền trực tiếp vào agent factory. [Quy tắc sắp xếp thứ tự cuối cùng](/oss/python/langchain/event-streaming#register-transformers-on-middleware) giữ cho `ToolCallTransformer` tích hợp sẵn ở vị trí đầu tiên và xếp các entry do người gọi cung cấp nằm ở cuối.

Bạn thiết lập thuộc tính class `transformers` bằng một tuple các factory callable. Mỗi factory có hình thức `Callable[[tuple[str, ...]], StreamTransformer]` và được gọi dưới dạng `factory(scope)`, trong đó `scope` là một tuple phạm vi mini-mux (`()` dành cho gốc, khác rỗng dành cho các subgraph); việc trả về một transformer mới cho mỗi lần gọi đảm bảo mỗi subgraph được cách ly hoàn toàn.

```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware


class ToolActivityMiddleware(AgentMiddleware):
    transformers = (ToolActivityTransformer,)


agent = create_agent(
    model="gpt-5-nano",
    tools=[...],
    middleware=[ToolActivityMiddleware()],
)
```

Xem mục [Đăng ký transformers trên middleware](/oss/python/langchain/event-streaming#register-transformers-on-middleware) để biết toàn bộ các quy tắc về thứ tự và ví dụ về redaction PII.

## Thứ tự thực thi

Khi sử dụng nhiều middleware cùng lúc, bạn cần nắm rõ cách chúng thực thi:

```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
agent = create_agent(
    model="gpt-5.5",
    middleware=[middleware1, middleware2, middleware3],
    tools=[...],
)
```

<Accordion title="Luồng thực thi">
  **Các before hook chạy theo thứ tự:**

  1. `middleware1.before_agent()`
  2. `middleware2.before_agent()`
  3. `middleware3.before_agent()`

  **Vòng lặp của Agent bắt đầu**

  4. `middleware1.before_model()`
  5. `middleware2.before_model()`
  6. `middleware3.before_model()`

  **Các wrap hook lồng nhau như khi gọi hàm:**

  7. `middleware1.wrap_model_call()` → `middleware2.wrap_model_call()` → `middleware3.wrap_model_call()` → model

  **Các after hook chạy theo chiều ngược lại:**

  8. `middleware3.after_model()`
  9. `middleware2.after_model()`
  10. `middleware1.after_model()`

  **Vòng lặp của Agent kết thúc**

  11. `middleware3.after_agent()`
  12. `middleware2.after_agent()`
  13. `middleware1.after_agent()`
</Accordion>

**Các quy tắc chính:**

* Các `before_*` hook: Từ đầu đến cuối
* Các `after_*` hook: Từ cuối lên đầu (đảo ngược)
* Các `wrap_*` hook: Lồng nhau (middleware đầu tiên sẽ bao bọc tất cả các middleware còn lại)

## Điểm nhảy (Agent jumps)

Để kết thúc sớm từ bên trong middleware, bạn cần trả về một dictionary chứa `jump_to`:

**Các đích nhảy (jump targets) khả dụng:**

* `'end'`: Nhảy đến cuối quá trình thực thi agent (hoặc hook `after_agent` đầu tiên)
* `'tools'`: Nhảy đến node tools
* `'model'`: Nhảy đến node model (hoặc hook `before_model` đầu tiên)

<Tabs>
  <Tab title="Decorator">
    ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
    from langchain.agents.middleware import after_model, hook_config, AgentState
    from langchain.messages import AIMessage
    from langgraph.runtime import Runtime
    from typing import Any


    @after_model
    @hook_config(can_jump_to=["end"])
    def check_for_blocked(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        last_message = state["messages"][-1]
        if "BLOCKED" in last_message.content:
            return {
                "messages": [AIMessage("Tôi không thể phản hồi yêu cầu đó.")],
                "jump_to": "end"
            }
        return None
    ```
  </Tab>
</Tabs>

## Cấu hình tracing

<Note>Yêu cầu `langchain>=1.3.15`.</Note>

Các span của middleware hook mặc định sẽ trace (theo dõi) thông tin input và output của chúng. Bạn thiết lập `trace_policy` để định hình những dữ liệu mà chúng ghi lại. `TracePolicy` chấp nhận các callable như `process_inputs` và `process_outputs` để biến đổi giá trị trace; `omit_payload` sẽ bỏ qua dữ liệu đó hoàn toàn. Điều này có thể hữu ích như một cách tối ưu hóa hiệu suất khi lịch sử message dài không mang lại thêm ý nghĩa cho chức năng của middleware.

Để bỏ qua payload input trong trace của middleware:

```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
from langchain.agents.middleware import AgentMiddleware, TracePolicy, omit_payload

class MyMiddleware(AgentMiddleware):
    trace_policy = TracePolicy(process_inputs=omit_payload)
```

Để áp dụng chính sách cho toàn bộ middleware, hãy cấu hình giá trị mặc định toàn cầu:

```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
from langchain.agents.middleware import configure_trace_policy, TracePolicy, omit_payload

configure_trace_policy(TracePolicy(process_inputs=omit_payload))  # truyền None để xóa
```

Cấu hình `trace_policy` riêng của một middleware sẽ ghi đè thiết lập mặc định toàn cầu.

## Best practices (Thực hành tốt nhất)

1. Giữ cho middleware tập trung - mỗi middleware chỉ nên làm tốt một việc duy nhất
2. Xử lý lỗi một cách khéo léo (gracefully) - đừng để lỗi trong middleware làm sập (crash) agent
3. **Sử dụng đúng loại hook**:
   * Node-style cho các logic mang tính tuần tự (logging, validation)
   * Wrap-style để kiểm soát luồng (retry, fallback, caching)
4. Ghi chú rõ ràng bất kỳ thuộc tính custom state nào
5. Unit test từng middleware một cách độc lập trước khi tích hợp
6. Cân nhắc đến thứ tự thực thi - đặt các middleware quan trọng lên đầu danh sách
7. Sử dụng các middleware tích hợp sẵn của framework khi có thể

## Các ví dụ

### Dynamic prompt (Prompt động)

Sửa đổi system prompt ở runtime một cách linh hoạt để tiêm context, các hướng dẫn cụ thể theo từng người dùng, hoặc các thông tin khác trước mỗi lời gọi model. Đây là một trong những use case phổ biến nhất của middleware.

Sử dụng trường `system_message` trên `ModelRequest` để đọc và chỉnh sửa system prompt. Trường này chứa một đối tượng [`SystemMessage`](https://reference.langchain.com/python/langchain-core/messages/system/SystemMessage) (ngay cả khi agent được tạo với tham số chuỗi string `system_prompt`).

<Tabs>
  <Tab title="Decorator">
    ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
    from collections.abc import Callable

    from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
    from langchain.messages import SystemMessage


    @wrap_model_call
    def add_context(
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        new_content = list(request.system_message.content_blocks) + [
            {"type": "text", "text": "Ngữ cảnh bổ sung."}
        ]
        new_system_message = SystemMessage(content=new_content)
        return handler(request.override(system_message=new_system_message))
    ```
  </Tab>
</Tabs>

<Note>
  * `ModelRequest.system_message` luôn luôn là một đối tượng [`SystemMessage`](https://reference.langchain.com/python/langchain-core/messages/system/SystemMessage), ngay cả khi khởi tạo agent bằng `system_prompt="string"`
  * Sử dụng `SystemMessage.content_blocks` để lấy cấu trúc content dưới dạng danh sách block, bất kể nội dung ban đầu là một chuỗi hay một danh sách
  * Khi thay đổi system message, hãy sử dụng `content_blocks` và gộp thêm (append) các block mới vào đuôi để giữ nguyên cấu trúc cũ
  * Bạn có thể truyền trực tiếp các đối tượng [`SystemMessage`](https://reference.langchain.com/python/langchain-core/messages/system/SystemMessage) vào tham số `system_prompt` của `create_agent` nếu cần xử lý cache control nâng cao
</Note>

### Lựa chọn model động

<Tabs>
  <Tab title="Decorator">
    ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
    from collections.abc import Callable

    from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
    from langchain.chat_models import init_chat_model

    complex_model = init_chat_model("claude-sonnet-4-6")
    simple_model = init_chat_model("claude-haiku-4-5-20251001")


    @wrap_model_call
    def dynamic_model(
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        if len(request.messages) > 10:
            model = complex_model
        else:
            model = simple_model
        return handler(request.override(model=model))
    ```
  </Tab>
</Tabs>

### Lựa chọn tool động

Bạn có thể lọc các tool liên quan trong thời gian runtime nhằm nâng cao hiệu suất và độ chính xác của agent. Phần này sẽ hướng dẫn lọc các tool đã đăng ký trước. Để tự động đăng ký các tool được khám phá tại runtime (ví dụ: qua MCP server), hãy xem [Runtime tool registration](/oss/python/langchain/tools#dynamic-tool-selection).

**Lợi ích mang lại:**

* **Prompt ngắn hơn** - Giảm thiểu sự phức tạp bằng cách chỉ hiển thị những tool liên quan
* **Độ chính xác cao hơn** - Model sẽ lựa chọn chính xác hơn khi có ít option
* **Kiểm soát quyền truy cập** - Lọc linh hoạt các tool dựa trên quyền truy cập của user hiện tại

<Tabs>
  <Tab title="Decorator">
    ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
    from langchain.agents import create_agent
    from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
    from typing import Callable


    @wrap_model_call
    def select_tools(
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        """Middleware để chọn các tool liên quan dựa trên state/ngữ cảnh."""
        # Chọn một tập con nhỏ các tool liên quan dựa trên state/ngữ cảnh
        relevant_tools = select_relevant_tools(request.state, request.runtime)
        return handler(request.override(tools=relevant_tools))

    agent = create_agent(
        model="gpt-5.5",
        tools=all_tools,  # Tất cả các tool khả dụng cần được đăng ký trước
        middleware=[select_tools],
    )
    ```
  </Tab>
</Tabs>

### Theo dõi lời gọi tool

<Tabs>
  <Tab title="Decorator">
    ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
    from collections.abc import Callable

    from langchain.agents.middleware import wrap_tool_call
    from langchain.messages import ToolMessage
    from langchain.tools.tool_node import ToolCallRequest
    from langgraph.types import Command


    @wrap_tool_call
    def monitor_tool(
        request: ToolCallRequest,
        handler: Callable[[ToolCallRequest], ToolMessage | Command],
    ) -> ToolMessage | Command:
        print(f"Đang thực thi tool: {request.tool_call['name']}")
        print(f"Tham số: {request.tool_call['args']}")
        try:
            result = handler(request)
            print("Tool đã hoàn thành thành công")
            return result
        except Exception as e:
            print(f"Tool thất bại: {e}")
            raise
    ```
  </Tab>
</Tabs>

### Caching prompt (Anthropic)

Khi làm việc với các model Anthropic, hãy sử dụng các content block có cấu trúc kèm theo cache control directives để lưu cache các system prompt có kích thước lớn:

<Tabs>
  <Tab title="Decorator">
    ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
    from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
    from langchain.messages import SystemMessage
    from typing import Callable


    @wrap_model_call
    def add_cached_context(
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        # Luôn làm việc với content_blocks
        new_content = list(request.system_message.content_blocks) + [
            {
                "type": "text",
                "text": "Dưới đây là một tài liệu lớn cần phân tích:\n\n<document>...</document>",
                # nội dung cho đến thời điểm này sẽ được cache lại
                "cache_control": {"type": "ephemeral"}
            }
        ]

        new_system_message = SystemMessage(content=new_content)
        return handler(request.override(system_message=new_system_message))
    ```
  </Tab>
</Tabs>

**Lưu ý:**

* `ModelRequest.system_message` luôn luôn là một đối tượng [`SystemMessage`](https://reference.langchain.com/python/langchain-core/messages/system/SystemMessage), ngay cả khi bạn tạo agent bằng tham số `system_prompt="string"`
* Hãy dùng `SystemMessage.content_blocks` để truy xuất nội dung dưới dạng danh sách các block, bất kể giá trị nội dung gốc truyền vào là string hay list
* Khi thay đổi các system message, bạn nên sử dụng `content_blocks` và chèn (append) các block mới vào để giữ cấu trúc đang có
* Bạn hoàn toàn có thể truyền thẳng các đối tượng [`SystemMessage`](https://reference.langchain.com/python/langchain-core/messages/system/SystemMessage) vào trong `create_agent` qua tham số `system_prompt` với các tác vụ nâng cao như thiết lập bộ nhớ đệm (cache control)

:::

## Tài nguyên bổ sung

* [Tham chiếu Middleware API](https://reference.langchain.com/python/langchain/middleware/)
* [Các middleware tích hợp sẵn](/oss/python/langchain/middleware/built-in)
* [Testing agents](/oss/python/langchain/test/)

***

<div className="source-links">
  <Callout icon="terminal-2">
    [Kết nối các tài liệu này](/use-these-docs) với Claude, VSCode và hơn thế nữa thông qua MCP để nhận câu trả lời theo thời gian thực.
  </Callout>

  <Callout icon="edit">
    [Chỉnh sửa trang này trên GitHub](https://github.com/langchain-ai/docs/edit/main/src/oss/langchain/middleware/custom.mdx) hoặc [báo cáo lỗi](https://github.com/langchain-ai/docs/issues/new/choose).
  </Callout>
</div>